# A Sample Classifier Comparison on Predicting Match Rank
If a given match has no recorded average rank, can we predict it based on individual player's performance? This sample notebook hypothesize three classifiers that can potentially do this jobs: Gaussian  Naive Bayes, Logistic Regression, and Decision Tree Classifiers.

## Content:
1. Pull from source database.
2. Select features and response.
3. Fit and evaluate multiple models and return a summary reports.

> Note: The data being used excludes matches where the last recorded stat snapshot was stale (unlogged_duration != 0) or where the game ended abnormally early (game_duration <= 300 seconds).

In [ ]:
USE WAREHOUSE COMPUTE_WH;

USE DATABASE LEAGUE_RECORDS;

USE SCHEMA GOLD;

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from snowflake.snowpark import DataFrame as SnowparkDataFrame

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    OneHotEncoder
)
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
pd.set_option('display.float_format', '{:.4f}'.format)

## Features and Response Schema
All are numeric stats recorded at match end per player.

* `LEVEL` int8
* `KILLS` int8
* `DEATHS` int8
* `ASSISTS` int8
* `CS` int16
* `TOTAL_GOLD` int32
* `GAME_DURATION` int16 (match length in seconds)
* `AVERAGE_RANK` object (average rank of players logged for a given match)

In [ ]:
SELECT *
FROM MATCHEND_PLAYER_STATS
JOIN MATCHEND_PIVOT_TEAMSTATS USING (MATCH_ID)
WHERE UNLOGGED_DURATION = 0 AND GAME_DURATION > 300
;

In [ ]:
SOURCE = SOURCE_sql.to_pandas()

In [ ]:
def feature_eng(src: pd.DataFrame) -> pd.DataFrame:
    return src[['LEVEL', 'KILLS', 'DEATHS', 'ASSISTS', 'CS', 'TOTAL_GOLD', 'GAME_DURATION', 'AVERAGE_RANK']]

In [ ]:
FEATURE_READY_SAMPLE = feature_eng(SOURCE).sample(1000)

In [ ]:
def data_quality(src: pd.DataFrame) -> pd.DataFrame:
    res = pd.DataFrame({
        'name': src.columns,
        'dtype': src.dtypes,
        'null': src.isna().mean(),
        'nunique': src.nunique() / len(src),
        'min': src.min(),
        'max': src.max(),
    })

    return res

data_quality(FEATURE_READY_SAMPLE)

## Data first glance:
1. No strong clustering behavior for the response column needing classifications.
2. `KILLS`, `DEATHS`, `ASSISTS` are 0-inflated.
3. `CS` are bimodal, reflecting supports not needing to farm gold.

In [ ]:
sns.pairplot(
    FEATURE_READY_SAMPLE,
    vars=['LEVEL', 'KILLS', 'DEATHS', 'ASSISTS', 'CS', 'TOTAL_GOLD', 'GAME_DURATION'],
    hue='AVERAGE_RANK',
    plot_kws={'alpha': 0.4, 's': 12}
)

In [ ]:
def find_classifier(
    src: pd.DataFrame,
    response: str,
    features: list[str],
    # -- ColumnTransformer
    onehot: list[str] = None,
    mms: list[str] = None,
    ss: list[str] = None,
    drop: list[str] = None,
    # -- Others
    seed: int = 42,
    **kwargs
):
    # -- 01. Feature engineer from source data
    data = feature_eng(src)
    y = data[response]
    X = data[features]

    # -- 02. Split into train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        stratify=y,
        test_size=0.3,
        random_state=seed
    )

    # -- 03. Preprocessing
    ct_steps = []
    if onehot is not None:
        oh_encoder = OneHotEncoder(
            sparse_output=False,
            drop='first',
            dtype='int',
            handle_unknown='error'
        )
        ct_steps.append(('oh', oh_encoder, onehot))
    if ss is not None:
        ct_steps.append(('ss', StandardScaler(), ss))
    if mms is not None:
        ct_steps.append(('mms', MinMaxScaler(), mms))
    if drop is not None:
        ct_steps.append(('drop', 'drop', drop))

    ct = ColumnTransformer(ct_steps, remainder='passthrough')
    ct.set_output(transform='pandas')

    # -- 04. Find the best classifier using cross evaluation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    clf_choices = [
        ('gnb', GaussianNB()),
        ('dt', DecisionTreeClassifier(random_state=seed)),
        ('logreg', LogisticRegression(max_iter=1000, random_state=seed))
    ]

    clf_res = {}
    for name, clf in clf_choices:
        steps = [('ct', ct), (name, clf)]
        pipeline = Pipeline(steps)
        clf_res[name] = cross_val_score(pipeline, X_train, y_train, cv=skf)

    return pd.DataFrame(clf_res).boxplot()

In [ ]:
find_classifier(
    SOURCE,
    response='AVERAGE_RANK',
    features=['LEVEL', 'KILLS', 'DEATHS', 'ASSISTS', 'CS', 'TOTAL_GOLD', 'GAME_DURATION'],
    ss=['TOTAL_GOLD', 'GAME_DURATION'],
    mms=['LEVEL', 'KILLS', 'DEATHS', 'ASSISTS', 'CS'],
    seed=42
)